# Django inspectdb, Mapping Existing Tables

## Introduction to inspectdb
---

In this lesson, you'll learn how to **generate Django models from existing database tables**.

This is essential when integrating Django with legacy databases or databases managed by other systems.

1. [The Legacy Database Scenario](#The-Legacy-Database-Scenario),
    - [When You Have Existing Tables](#When-You-Have-Existing-Tables),
    - [The inspectdb Approach](#The-inspectdb-Approach),
2. [Setting Up a Legacy Database](#Setting-Up-a-Legacy-Database),
    - [Restore from .bak Backup](#Restore-from-.bak-Backup),
    - [Adding a Second Database to Django](#Adding-a-Second-Database-to-Django),
3. [Using inspectdb](#Using-inspectdb),
    - [Basic Usage](#Basic-Usage),
    - [Inspecting Specific Tables](#Inspecting-Specific-Tables),
4. [Editing Generated Models](#Editing-Generated-Models),
    - [What to Always Check](#What-to-Always-Check),
    - [db_table and db_column](#db_table-and-db_column),
5. [Relations and Foreign Keys](#Relations-and-Foreign-Keys),
    - [How inspectdb Recognizes Relations](#How-inspectdb-Recognizes-Relations),
    - [Fixing Relation Issues](#Fixing-Relation-Issues),
6. [Testing the Models](#Testing-the-Models),
7. [🧠 EXERCISE 🧠, Generate Models from Legacy Database](#🧠-EXERCISE-🧠,-Generate-Models-from-Legacy-Database).

<br>

## The Legacy Database Scenario

---

### When You Have Existing Tables

---

Common scenarios where you need to work with existing databases:

<br>

| Scenario | Description |
|----------|-------------|
| **Legacy migration** | Old PHP/Java app being replaced with Django |
| **Shared database** | Multiple apps use the same database |
| **Data warehouse** | Reporting database managed by DBA team |
| **Third-party system** | Database from vendor software |
| **Microservices** | Service needs read access to another service's DB |

<br>

### The inspectdb Approach

---

Django provides `inspectdb` command that:

1. **Connects** to your database
2. **Reads** table schemas
3. **Generates** Django model code

<br>

```
┌─────────────────────┐      inspectdb      ┌─────────────────────┐
│   Existing Tables   │  ─────────────────► │   Django Models     │
│   (SQL Schema)      │                     │   (Python Code)     │
└─────────────────────┘                     └─────────────────────┘
```

<br>

**Important:** `inspectdb` generates a *starting point* — you'll always need to review and edit the generated code!

<br>

## Setting Up a Legacy Database

---

For this lesson, we'll use a **real database backup** (`.bak` file) as our "legacy" database — the same way you'd integrate Django with an existing production database.

<br>

### Restore from .bak Backup

---

Copy the backup file into the container and restore it as a separate `legacy_db` database:

```bash
# EXAMPLE:
DELETE FROM blog_comment;
DELETE FROM blog_reviews;
DELETE FROM blog_blog;

# 1. Fix file ownership so MSSQL can read it
docker compose exec -u root db bash -c \
  "cp /var/opt/mssql/backup/blog_db.bak /tmp/blog_db.bak && chown mssql:mssql /tmp/blog_db.bak"

# Check the file inside of your container
docker compose exec db ls -lh /tmp/blog_db.bak

# 2. Restore as legacy_db (blog_db stays untouched)
docker compose exec db /opt/mssql-tools18/bin/sqlcmd \
  -S localhost -U SA -P "Blog_pass1" -No \
  -Q "RESTORE DATABASE [legacy_db]
      FROM DISK = N'/tmp/blog_db.bak'
      WITH MOVE 'blog_db'     TO '/var/opt/mssql/data/legacy_db.mdf',
           MOVE 'blog_db_log' TO '/var/opt/mssql/data/legacy_db_log.ldf',
           REPLACE;"

# 3. Verify tables
docker compose exec db /opt/mssql-tools18/bin/sqlcmd \
  -S localhost -U SA -P "Blog_pass1" -No -d legacy_db \
  -Q "SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_TYPE='BASE TABLE' ORDER BY TABLE_NAME;"
```

> **Note:** Files copied via `docker cp` are owned by the host user (UID 501) — MSSQL cannot read them. The `chown mssql:mssql` inside the container fixes the ownership.

<br>

### Adding a Second Database to Django

---

Add `LEGACY_DB_NAME` to `.env`:

```ini
# .env
DB_ENGINE=mssql
DB_NAME=blog_db
DB_USER=SA
DB_PASSWORD=Blog_pass1
DB_HOST=localhost
DB_PORT=1433
DB_DRIVER=ODBC Driver 18 for SQL Server

LEGACY_DB_NAME=legacy_db   # <-- add this
```

<br>

Add a `legacy` entry to `DATABASES` in `settings.py`:

```python
# mysite/mysite/settings.py

if os.environ.get('LEGACY_DB_NAME'):
    DATABASES['legacy'] = {
        'ENGINE': os.environ.get('DB_ENGINE', 'mssql'),
        'NAME': os.environ.get('LEGACY_DB_NAME'),
        'USER': os.environ.get('DB_USER', ''),
        'PASSWORD': os.environ.get('DB_PASSWORD', ''),
        'HOST': os.environ.get('DB_HOST', ''),
        'PORT': os.environ.get('DB_PORT', ''),
        'OPTIONS': {
            'driver': os.environ.get('DB_DRIVER', ''),
            'extra_params': 'TrustServerCertificate=yes',
        },
    }
```

<br>

Now run `inspectdb` against the legacy database using `--database legacy`:

```bash
# Preview in terminal
python manage.py inspectdb --database legacy blog_blog blog_comment blog_reviews

# Save to file
python manage.py inspectdb --database legacy blog_blog blog_comment blog_reviews \
  > blog/models_legacy.py
```

> **Why a separate database entry?** Using `--database legacy` lets you inspect the legacy DB without touching your main `blog_db` configuration.

<br>

## Using `inspectdb`

---

### Basic Usage

---

The basic syntax is:

```bash
python manage.py inspectdb --database legacy
```

This outputs Django model code to stdout. To save it:

```bash
# Inspect specific tables and save
python manage.py inspectdb --database legacy blog_blog blog_comment blog_reviews \
  > blog/models_legacy.py

# Or preview first
python manage.py inspectdb --database legacy blog_blog blog_comment blog_reviews | less
```

<br>

**Real output of `inspectdb --database legacy blog_blog blog_comment blog_reviews`:**

In [ ]:
# This is an auto-generated Django model module.
# You'll have to do the following manually to clean this up:
#   * Rearrange models' order
#   * Make sure each model has one field with primary_key=True
#   * Make sure each ForeignKey and OneToOneField has `on_delete` set to the desired behavior
#   * Remove `managed = False` lines if you wish to allow Django to create, modify, and delete the table
# Feel free to rename the models, but don't rename db_table values or field names.
from django.db import models


class BlogBlog(models.Model):
    id = models.BigAutoField(primary_key=True)
    title = models.CharField(max_length=200, db_collation='SQL_Latin1_General_CP1_CI_AS')
    author = models.CharField(max_length=100, db_collation='SQL_Latin1_General_CP1_CI_AS')
    published_date = models.DateField()
    author_email = models.CharField(max_length=254, db_collation='SQL_Latin1_General_CP1_CI_AS')
    category_type = models.SmallIntegerField()
    content = models.TextField(db_collation='SQL_Latin1_General_CP1_CI_AS')
    slug = models.CharField(max_length=200, db_collation='SQL_Latin1_General_CP1_CI_AS')

    class Meta:
        managed = False
        db_table = 'blog_blog'


class BlogComment(models.Model):
    id = models.BigAutoField(primary_key=True)
    content = models.TextField(db_collation='SQL_Latin1_General_CP1_CI_AS')
    created_at = models.DateTimeField()
    updated_at = models.DateTimeField()
    blog = models.ForeignKey(BlogBlog, models.DO_NOTHING)

    class Meta:
        managed = False
        db_table = 'blog_comment'


class BlogReviews(models.Model):
    id = models.BigAutoField(primary_key=True)
    rating = models.SmallIntegerField()
    comment = models.TextField(db_collation='SQL_Latin1_General_CP1_CI_AS')
    created_at = models.DateTimeField()
    blog = models.ForeignKey(BlogBlog, models.DO_NOTHING)
    user = models.ForeignKey('AuthUser', models.DO_NOTHING)

    class Meta:
        managed = False
        db_table = 'blog_reviews'
        unique_together = (('blog', 'user'),)


<br>

### Inspecting Specific Tables

---

You can inspect only specific tables by passing their names:

```bash
# Single table
python manage.py inspectdb --database legacy blog_blog

# Multiple tables
python manage.py inspectdb --database legacy blog_blog blog_comment blog_reviews

# All tables (beware — includes Django system tables: auth_*, django_*)
python manage.py inspectdb --database legacy
```

<br>

## Editing Generated Models

---

### What to Always Check

---

The generated code is a **starting point**. Always review and fix:

<br>

| Issue | inspectdb generates | Fix |
|-------|---------------------|-----|
| **Model names** | `BlogBlog`, `BlogReviews` | Rename to `Blog`, `BlogReview` |
| **`db_collation`** | `db_collation='SQL_Latin1_General_CP1_CI_AS'` | Remove — no effect with `managed = False` |
| **`on_delete`** | Always `DO_NOTHING` | Change to `CASCADE`, `PROTECT`, etc. |
| **FK to `auth_user`** | `ForeignKey('AuthUser', ...)` | Replace with `ForeignKey(User, ...)` |
| **Field types** | `CharField` for email, `SmallIntegerField` for choices | Use `EmailField`, `PositiveSmallIntegerField` |
| **`managed`** | `managed = False` | Keep if table is external; remove if Django should manage it |
| **Meta options** | Minimal | Add `ordering`, `indexes`, `verbose_name` as needed |

`db_collation='SQL_Latin1_General_CP1_CI_AS'`

This is **an MSSQL-specific** value — it tells the database how to sort and compare strings (e.g., case-insensitive, Latin1).

You can safely delete this from your Django models; it is only informative for cases where Django manages the table. Since you have `managed = False`, it has no practical effect.

```python
# Generated by inspectdb:
title = models.CharField(max_length=200, db_collation='SQL_Latin1_General_CP1_CI_AS')

# After cleanup:
title = models.CharField(max_length=200)
```

`Fixing ForeignKeys (User Model)`

`inspectdb` named the model based on the table name `auth_user` → `AuthUser`. If you want the `ForeignKey` to point directly to the built-in Django `User` model, replace the generated code:

```python
# Generated by inspectdb:
user = models.ForeignKey('AuthUser', models.DO_NOTHING)

# After cleanup — use the built-in User model:
from django.contrib.auth.models import User

user = models.ForeignKey(User, on_delete=models.CASCADE)
```

`managed = False`

This tells Django: "This table exists in the database, but do not touch it—no migrate, no DROP TABLE."

Keep managed = False if the table is managed externally (e.g., by a legacy system or a DBA team).

Delete it (or set `managed = True`) if you want Django to manage the table via migrations. Be careful: if you switch to True, make sure your models perfectly match the database to avoid overwriting or corrupting existing data.

<br>

**Cleaned up models (based on real `blog_blog`, `blog_comment`, `blog_reviews` tables):**

In [ ]:
# blog/models_legacy.py (after cleanup)

from django.db import models
from django.contrib.auth.models import User


class CategoryType(models.IntegerChoices):
    TECH      = 1, 'Technology'
    BUSINESS  = 2, 'Business'
    LIFESTYLE = 3, 'Lifestyle'
    NEWS      = 4, 'News'


class Blog(models.Model):
    title = models.CharField(max_length=200)           # removed db_collation
    author = models.CharField(max_length=100)
    published_date = models.DateField()
    author_email = models.EmailField(max_length=254)   # EmailField instead of CharField
    category_type = models.PositiveSmallIntegerField(  # better field type + choices
        choices=CategoryType.choices,
        default=CategoryType.TECH,
    )
    content = models.TextField(blank=True)
    slug = models.SlugField(max_length=200, blank=True)

    class Meta:
        managed = False          # Django won't touch this table
        db_table = 'blog_blog'   # maps to the real table name
        ordering = ['-published_date', 'title']

    def __str__(self):
        return self.title


class Comment(models.Model):
    content = models.TextField()
    created_at = models.DateTimeField()
    updated_at = models.DateTimeField()
    blog = models.ForeignKey(
        Blog, on_delete=models.CASCADE,  # DO_NOTHING → CASCADE
        related_name='comments'
    )

    class Meta:
        managed = False
        db_table = 'blog_comment'
        ordering = ['-created_at']

    def __str__(self):
        return f'Comment on {self.blog.title}'


class BlogReview(models.Model):                        # singular, not BlogReviews
    rating = models.PositiveSmallIntegerField()        # PositiveSmallIntegerField
    comment = models.TextField(blank=True)
    created_at = models.DateTimeField()
    blog = models.ForeignKey(
        Blog, on_delete=models.CASCADE,
        related_name='reviews'
    )
    user = models.ForeignKey(
        User, on_delete=models.CASCADE,  # 'AuthUser' → User
        related_name='blog_reviews'
    )

    class Meta:
        managed = False
        db_table = 'blog_reviews'
        unique_together = [['blog', 'user']]

    def __str__(self):
        return f'Review {self.rating}/5 for {self.blog.title}'


<br>

### `db_table` and `db_column`

---

These options map Django models/fields to existing database names:

<br>

**`db_table`** — maps model to a specific table name:

```python
class Blog(models.Model):  # but w/ BlogBlog --> blog_blogblog table
    # ...
    class Meta:
        db_table = 'blog_blog'  # actual table name in DB
```

<br>

**`db_column`** — maps a field to a specific column name (useful when you rename a field in Python):

```python
class BlogReview(models.Model):
    # Python name is 'post', but DB column is 'blog_id'
    post = models.ForeignKey(
        Blog,
        db_column='blog_id',   # actual column name in DB
        on_delete=models.CASCADE
    )
```

<br>

**When to use:**

| Situation | Solution |
|-----------|----------|
| Table name differs from Django convention | Use `db_table` |
| Column name differs from Django convention | Use `db_column` |
| Want nicer Python field names | Rename field + add `db_column` with original name |
| Primary key isn't `id` | Keep original field with `primary_key=True` |

<br>

## Relations and Foreign Keys

---

### How `inspectdb` Recognizes Relations

---

`inspectdb` detects foreign keys from the database schema:

```sql
-- SQL foreign key in blog_comment:
blog_id BIGINT REFERENCES blog_blog(id)

-- Becomes:
blog = models.ForeignKey(BlogBlog, models.DO_NOTHING)
```

<br>

**What inspectdb detects:**

| Database constraint | Django result |
|---------------------|---------------|
| `FOREIGN KEY` | `ForeignKey` with `DO_NOTHING` |
| `UNIQUE` on FK column | `OneToOneField` |
| Composite `UNIQUE` (e.g. `blog + user`) | `unique_together` in Meta |
| FK to `auth_user` | `ForeignKey('AuthUser', ...)` — string reference |

<br>

### Fixing Relation Issues

---

**Issue 1: `DO_NOTHING` is dangerous**

```bash
# inspectdb generates:
blog = models.ForeignKey(BlogBlog, models.DO_NOTHING)

# Fix:
blog = models.ForeignKey(Blog, on_delete=models.CASCADE)   # delete comment when blog deleted
# or
blog = models.ForeignKey(Blog, on_delete=models.PROTECT)   # prevent blog deletion if comments exist
```

<br>

| on_delete | Behavior |
|-----------|----------|
| `CASCADE` | Delete related objects when parent is deleted |
| `PROTECT` | Prevent deletion if related objects exist |
| `SET_NULL` | Set FK to NULL (requires `null=True`) |
| `DO_NOTHING` | Do nothing — can cause database integrity errors! |

<br>

**Issue 2: FK to `auth_user` — use Django's built-in `User` model**

```bash
# inspectdb generates:
user = models.ForeignKey('AuthUser', models.DO_NOTHING)

# Fix:
from django.contrib.auth.models import User

user = models.ForeignKey(User, on_delete=models.CASCADE, related_name='blog_reviews')
```

> `'AuthUser'` is a string reference to an auto-generated class. Since Django already provides `django.contrib.auth.models.User`, always replace it with the real import.

<br>

**Issue 3: `db_collation` — remove it**

```python
# inspectdb generates (MSSQL-specific):
title = models.CharField(max_length=200, db_collation='SQL_Latin1_General_CP1_CI_AS')

# Fix — remove db_collation, it has no effect with managed = False:
title = models.CharField(max_length=200)
```

<br>

## Testing the Models

---

After cleaning up the models, test them in Django shell:

```bash
python manage.py shell
```

```python
>>> from blog.models_legacy import Blog, Comment, BlogReview

>>> # List all blogs from legacy DB
>>> Blog.objects.using('legacy').all()
<QuerySet [<Blog: Django Bootcamp>, ...]>

>>> # Get blog with its comments
>>> blog = Blog.objects.using('legacy').first()
>>> blog.comments.using('legacy').all()
<QuerySet [<Comment: Comment on Django Bootcamp>]>

>>> # Get reviews with user info
>>> BlogReview.objects.using('legacy').select_related('blog', 'user').all()
<QuerySet [<BlogReview: Review 5/5 for Django Bootcamp>]>
```

> **Note:** Use `.using('legacy')` to query the legacy database explicitly. Without it, Django queries the `default` database.

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **inspectdb** | Generates Django models from existing database tables |
| **`--database`** | `inspectdb --database legacy` — inspect a specific DB |
| **Specific tables** | `inspectdb --database legacy blog_blog blog_comment` |
| **`db_table`** | Maps model to table: `db_table = 'blog_blog'` |
| **`db_column`** | Maps field to column: `db_column = 'blog_id'` |
| **`managed = False`** | Django won't create/alter/drop the table |
| **Always fix** | `DO_NOTHING` → `CASCADE`/`PROTECT`, `'AuthUser'` → `User`, remove `db_collation` |
| **`.using('legacy')`** | Query a non-default database in Django shell |

<br>

### 🧠 EXERCISE 🧠, Generate Models from Legacy Database

---

Using the `legacy_db` you restored from `blog_db.bak`:

1. Add `LEGACY_DB_NAME=legacy_db` to `.env`
2. Add `DATABASES['legacy']` to `settings.py`
3. Run `inspectdb` for the blog tables:
```bash
python manage.py inspectdb --database legacy blog_blog blog_comment blog_reviews \
  > blog/models_legacy.py
```
4. Edit `blog/models_legacy.py`:
   - Rename `BlogBlog` → `Blog`, `BlogComment` → `Comment`, `BlogReviews` → `BlogReview`
   - Remove all `db_collation=...` arguments
   - Fix `on_delete` from `DO_NOTHING` to `CASCADE`
   - Replace `ForeignKey('AuthUser', ...)` with `ForeignKey(User, ...)`
   - Improve field types: `EmailField`, `PositiveSmallIntegerField`
   - Add `__str__` methods
5. Test in Django shell using `.using('legacy')`

<details>
    <summary>▶️ Solution</summary>

**3. Generate models:**

```bash
python manage.py inspectdb --database legacy blog_blog blog_comment blog_reviews > blog/models_legacy.py
```

**4. Edit `blog/models_legacy.py`:**

```python
from django.db import models
from django.contrib.auth.models import User


class Blog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    published_date = models.DateField()
    author_email = models.EmailField(max_length=254)
    category_type = models.PositiveSmallIntegerField()
    content = models.TextField(blank=True)
    slug = models.SlugField(max_length=200, blank=True)

    class Meta:
        managed = False
        db_table = 'blog_blog'
        ordering = ['-published_date', 'title']

    def __str__(self):
        return self.title


class Comment(models.Model):
    content = models.TextField()
    created_at = models.DateTimeField()
    updated_at = models.DateTimeField()
    blog = models.ForeignKey(Blog, on_delete=models.CASCADE, related_name='comments')

    class Meta:
        managed = False
        db_table = 'blog_comment'
        ordering = ['-created_at']

    def __str__(self):
        return f'Comment on {self.blog.title}'


class BlogReview(models.Model):
    rating = models.PositiveSmallIntegerField()
    comment = models.TextField(blank=True)
    created_at = models.DateTimeField()
    blog = models.ForeignKey(Blog, on_delete=models.CASCADE, related_name='reviews')
    user = models.ForeignKey(User, on_delete=models.CASCADE, related_name='blog_reviews')

    class Meta:
        managed = False
        db_table = 'blog_reviews'
        unique_together = [['blog', 'user']]

    def __str__(self):
        return f'Review {self.rating}/5 for {self.blog.title}'
```

**5. Test in shell:**

```python
>>> from blog.models_legacy import Blog, Comment, BlogReview
>>> Blog.objects.using('legacy').all()
>>> Comment.objects.using('legacy').select_related('blog').all()
>>> BlogReview.objects.using('legacy').select_related('blog', 'user').all()
```

</details>

---